In [2]:
import torch
import string

In [3]:
words = open('names.txt', 'r').read().splitlines()

In [4]:
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [5]:
cleaned_words = []

for word in words:
    cleaned_words.append('..' + word + '.')

cleaned_words[:10]

['..emma.',
 '..olivia.',
 '..ava.',
 '..isabella.',
 '..sophia.',
 '..charlotte.',
 '..mia.',
 '..amelia.',
 '..harper.',
 '..evelyn.']

In [6]:
for w in cleaned_words[:1]:
    for i in range(2, len(w)):
        print(w[i-2], w[i-1], w[i])

. . e
. e m
e m m
m m a
m a .


In [7]:
stoi = {c:i for i, c in enumerate(string.ascii_lowercase)} | {'.': 26}
itos = {i:c for i, c in enumerate(string.ascii_lowercase)} | {26: '.'}

In [8]:
xs = []
ys = []


for w in cleaned_words:
    for i in range(2, len(w)):
        xs.append(stoi[w[i-2]] * 27 + stoi[w[i-1]])
        ys.append(stoi[w[i]])

ys = torch.tensor(ys)
print(xs[:10])
print(ys[:10])

[728, 706, 120, 336, 324, 728, 716, 389, 305, 237]
tensor([ 4, 12, 12,  0, 26, 14, 11,  8, 21,  8])


In [9]:
import torch.nn.functional as F

xenc = F.one_hot(torch.tensor(xs), num_classes=27*27).float()
xenc[:10]

tensor([[0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [10]:
xenc.shape

torch.Size([228146, 729])

In [11]:
g = torch.Generator().manual_seed(21)
W = torch.randn((729, 27), generator=g, requires_grad=True)

In [20]:
learning_rate = 100

# training loop
for k in range(200):

    # forward pass
    logits = xenc @ W
    counts = torch.exp(logits)
    prob = counts / counts.sum(1, keepdim=True)
    loss = - prob[torch.arange(len(xs)), ys].log().mean()

    # print loss
    print(loss.item())

    # backprop
    W.grad = None
    loss.backward()

    # update W
    W.data -= learning_rate * W.grad



2.2243216037750244
2.224274158477783
2.224226951599121
2.224179983139038
2.224132776260376
2.224086046218872
2.224039316177368
2.2239928245544434
2.2239463329315186
2.2238998413085938
2.223853826522827
2.2238075733184814
2.223761558532715
2.2237157821655273
2.22367000579834
2.2236242294311523
2.223578929901123
2.2235333919525146
2.2234883308410645
2.223443031311035
2.223397970199585
2.223353147506714
2.2233083248138428
2.2232635021209717
2.2232189178466797
2.223174571990967
2.223130226135254
2.223085880279541
2.2230417728424072
2.2229979038238525
2.2229537963867188
2.222910165786743
2.2228662967681885
2.222822666168213
2.2227792739868164
2.22273588180542
2.2226927280426025
2.222649574279785
2.222606658935547
2.2225635051727295
2.222520589828491
2.222478151321411
2.222435235977173
2.222393035888672
2.222350597381592
2.2223081588745117
2.22226619720459
2.222223997116089
2.222181797027588
2.222140073776245
2.2220983505249023
2.2220566272735596
2.222015142440796
2.2219738960266113
2.221932

In [23]:
g = torch.Generator().manual_seed(2147483647)

# name generator
for _ in range(20):
    word = ''
    i, j = 26, 26
    next_char = ''

    while next_char != 26:
        val = i * 27 + j
        x = F.one_hot(torch.tensor([val]), num_classes=27*27).float()
        p = (x @ W).exp()
        p = p / p.sum(1, keepdim=True)
        next_char = int(torch.multinomial(p, num_samples=1, generator=g).item())
        word += itos[next_char]
        i, j = j, next_char

    print(word[:-1])
    
    

    


dey
na
khilvill
rilayian
marni
obennah
el
lia
sa
maydivikuston
puntand
jario
valey
ae
kalaine
kate
violawhi
zarlettaiddielieus
mata
keesee
